In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
"""
Emotion Classification with IndicBERTv2
========================================
- Train  : English tweets  (tweet_emotions.xlsx)
- Test   : Assamese tweets (tweet_emotions_assamese.xlsx)
- Model  : ai4bharat/IndicBERTv2-MLM-Sam-TLM (HuggingFace)
- Metrics: Accuracy, Precision, Recall, Macro-F1,
           Confusion Matrix, Per-class ROC-AUC curves
"""

import os, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
SEED            = 42
MODEL_NAME      = "ai4bharat/IndicBERTv2-MLM-Sam-TLM"
MAX_LEN         = 128
BATCH_SIZE      = 32
EPOCHS          = 5
LR              = 2e-5
WARMUP_RATIO    = 0.1
DROPOUT         = 0.3
ENGLISH_PATH    = "/kaggle/input/datasets/dipaksarkar1/english-dataset/tweet_emotions.xlsx"
ASSAMESE_PATH   = "/kaggle/input/datasets/dipaksarkar1/dataassamese/tweet_emotions_assamese.xlsx"
OUTPUT_DIR      = "/mnt/user-data/outputs"
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs(OUTPUT_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")


# ─────────────────────────────────────────────────────────────
# 1. DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    df = df[["content", "emotion"]].dropna()
    df["content"] = df["content"].astype(str).str.strip()
    return df

print("\n[1/7] Loading data ...")
en_df  = load_data(ENGLISH_PATH)
asm_df = load_data(ASSAMESE_PATH)

# Encode labels on English data; reuse same encoder for Assamese
le = LabelEncoder()
en_df["label"]  = le.fit_transform(en_df["emotion"])
asm_df["label"] = le.transform(asm_df["emotion"])

NUM_CLASSES = len(le.classes_)
CLASS_NAMES = list(le.classes_)
print(f"  Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"  English rows : {len(en_df)}")
print(f"  Assamese rows: {len(asm_df)}")

# Train / Val split on English; full Assamese as test
train_df, val_df = train_test_split(
    en_df, test_size=0.15, stratify=en_df["label"], random_state=SEED
)
test_df = asm_df.copy()

print(f"  Train={len(train_df)} | Val={len(val_df)} | Test(Assamese)={len(test_df)}")

# Class weights for imbalanced data
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label"].values,
)
class_weights = torch.tensor(cw, dtype=torch.float).to(DEVICE)
print(f"  Class weights: {np.round(cw, 2)}")


# ─────────────────────────────────────────────────────────────
# 2. TOKENIZER & DATASET
# ─────────────────────────────────────────────────────────────
print("\n[2/7] Loading tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts.tolist()
        self.labels    = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_len,
            padding        = "max_length",
            truncation     = True,
            return_tensors = "pt",
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "label"          : torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = EmotionDataset(train_df["content"].values, train_df["label"].values, tokenizer, MAX_LEN)
val_ds   = EmotionDataset(val_df["content"].values,   val_df["label"].values,   tokenizer, MAX_LEN)
test_ds  = EmotionDataset(test_df["content"].values,  test_df["label"].values,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


# ─────────────────────────────────────────────────────────────
# 3. MODEL ARCHITECTURE
# ─────────────────────────────────────────────────────────────
class IndicBERTClassifier(nn.Module):
    """
    IndicBERTv2 encoder  →  [CLS] token  →  Dropout  →  MLP head
    """
    def __init__(self, model_name: str, num_classes: int, dropout: float):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        out    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Use [CLS] representation
        cls    = out.last_hidden_state[:, 0, :]
        cls    = self.dropout(cls)
        logits = self.classifier(cls)
        return logits

print("\n[3/7] Building model ...")
model = IndicBERTClassifier(MODEL_NAME, NUM_CLASSES, DROPOUT).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params    : {total_params:,}")
print(f"  Trainable params: {trainable:,}")


# ─────────────────────────────────────────────────────────────
# 4. OPTIMIZER, SCHEDULER, LOSS
# ─────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler    = GradScaler(enabled=DEVICE.type == "cuda")


# ─────────────────────────────────────────────────────────────
# 5. TRAINING & VALIDATION LOOP
# ─────────────────────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        labs  = batch["label"].to(DEVICE)
        optimizer.zero_grad()
        with autocast(enabled=DEVICE.type == "cuda"):
            logits = model(ids, mask)
            loss   = criterion(logits, labs)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * ids.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labs).sum().item()
        total   += ids.size(0)
    return total_loss / total, correct / total


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            labs  = batch["label"].to(DEVICE)
            with autocast(enabled=DEVICE.type == "cuda"):
                logits = model(ids, mask)
                loss   = criterion(logits, labs)
            total_loss += loss.item() * ids.size(0)
            probs  = torch.softmax(logits, dim=1)
            preds  = probs.argmax(dim=1)
            correct += (preds == labs).sum().item()
            total   += ids.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return (
        total_loss / total,
        correct / total,
        np.array(all_preds),
        np.array(all_labels),
        np.array(all_probs),
    )


print("\n[4/7] Training ...")
best_val_f1   = 0.0
history       = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler)
    vl_loss, vl_acc, vl_preds, vl_labels, _ = eval_epoch(model, val_loader, criterion)
    vl_f1 = f1_score(vl_labels, vl_preds, average="macro", zero_division=0)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    history["val_f1"].append(vl_f1)

    print(f"  Epoch {epoch}/{EPOCHS} | "
          f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
          f"Val Loss={vl_loss:.4f} Acc={vl_acc:.4f} F1={vl_f1:.4f}")

    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pt"))
        print(f"    ✓ Best model saved (Val F1={best_val_f1:.4f})")


# ─────────────────────────────────────────────────────────────
# 6. TEST EVALUATION ON ASSAMESE DATA
# ─────────────────────────────────────────────────────────────
print("\n[5/7] Evaluating on Assamese test set ...")
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pt"), map_location=DEVICE))
_, _, test_preds, test_labels, test_probs = eval_epoch(model, test_loader, criterion)

acc  = accuracy_score(test_labels, test_preds)
prec = precision_score(test_labels, test_preds, average="macro", zero_division=0)
rec  = recall_score(test_labels, test_preds, average="macro", zero_division=0)
f1   = f1_score(test_labels, test_preds, average="macro", zero_division=0)

# ROC-AUC (one-vs-rest, macro)
test_labels_bin = label_binarize(test_labels, classes=list(range(NUM_CLASSES)))
try:
    roc_auc = roc_auc_score(test_labels_bin, test_probs, average="macro", multi_class="ovr")
except Exception:
    roc_auc = float("nan")

print("\n" + "="*55)
print("  TEST RESULTS ON ASSAMESE DATA")
print("="*55)
print(f"  Accuracy       : {acc:.4f}")
print(f"  Precision (mac): {prec:.4f}")
print(f"  Recall (mac)   : {rec:.4f}")
print(f"  Macro F1       : {f1:.4f}")
print(f"  ROC-AUC (macro): {roc_auc:.4f}")
print("="*55)
print("\nPer-class Report:\n")
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, zero_division=0))


# ─────────────────────────────────────────────────────────────
# 7. PLOTS
# ─────────────────────────────────────────────────────────────
print("[6/7] Generating plots ...")

# --- 7a. Training curves ----------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history["train_loss"], "b-o", label="Train Loss")
axes[0].plot(epochs_range, history["val_loss"],   "r-o", label="Val Loss")
axes[0].set_title("Loss Curve", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_acc"], "b-o", label="Train Acc")
axes[1].plot(epochs_range, history["val_acc"],   "r-o", label="Val Acc")
axes[1].plot(epochs_range, history["val_f1"],    "g-s", label="Val F1")
axes[1].set_title("Accuracy & F1 Curve", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("IndicBERTv2 – Training History (English Data)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.close()

# --- 7b. Confusion Matrix (Test / Assamese) --------------------
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.4, ax=ax
)
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("True Label", fontsize=12)
ax.set_title(
    "Confusion Matrix – Assamese Test Set\n(IndicBERTv2, trained on English)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.close()

# --- 7c. ROC-AUC Curves (One-vs-Rest) --------------------------
from sklearn.metrics import roc_curve, auc as sk_auc

# Compute per-class ROC
fpr_dict, tpr_dict, roc_auc_dict = {}, {}, {}
for i in range(NUM_CLASSES):
    fpr_dict[i], tpr_dict[i], _ = roc_curve(test_labels_bin[:, i], test_probs[:, i])
    roc_auc_dict[i] = sk_auc(fpr_dict[i], tpr_dict[i])

# Macro-average ROC
all_fpr = np.unique(np.concatenate([fpr_dict[i] for i in range(NUM_CLASSES)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_CLASSES):
    mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
mean_tpr /= NUM_CLASSES
macro_auc = sk_auc(all_fpr, mean_tpr)

# Plot grid: 4 rows × 4 cols
cols = 4
rows = (NUM_CLASSES + 1 + cols - 1) // cols   # +1 for macro
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4.5, rows * 3.5))
axes = axes.flatten()

palette = plt.cm.tab20.colors
for i in range(NUM_CLASSES):
    ax = axes[i]
    ax.plot(fpr_dict[i], tpr_dict[i], color=palette[i % len(palette)],
            lw=2, label=f"AUC = {roc_auc_dict[i]:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    ax.set_title(CLASS_NAMES[i], fontsize=10, fontweight="bold")
    ax.set_xlabel("FPR", fontsize=8); ax.set_ylabel("TPR", fontsize=8)
    ax.tick_params(labelsize=7); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Macro average in last used panel
ax = axes[NUM_CLASSES]
ax.plot(all_fpr, mean_tpr, "navy", lw=2.5, linestyle="--",
        label=f"Macro AUC = {macro_auc:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
ax.set_title("Macro-Average ROC", fontsize=10, fontweight="bold")
ax.set_xlabel("FPR", fontsize=8); ax.set_ylabel("TPR", fontsize=8)
ax.tick_params(labelsize=7); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Hide remaining empty panels
for j in range(NUM_CLASSES + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    "ROC-AUC Curves – Assamese Test Set (One-vs-Rest)\nIndicBERTv2 trained on English",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_auc_curves.png"), dpi=150, bbox_inches="tight")
plt.close()

# --- 7d. Per-class ROC-AUC bar chart ---------------------------
fig, ax = plt.subplots(figsize=(12, 5))
auc_vals = [roc_auc_dict[i] for i in range(NUM_CLASSES)]
bars = ax.bar(CLASS_NAMES, auc_vals, color=palette[:NUM_CLASSES], edgecolor="black", linewidth=0.6)
ax.axhline(macro_auc, color="red", linestyle="--", linewidth=1.5, label=f"Macro Avg = {macro_auc:.3f}")
ax.set_ylim(0, 1.05)
ax.set_xlabel("Emotion Class"); ax.set_ylabel("ROC-AUC Score")
ax.set_title("Per-Class ROC-AUC – Assamese Test Set", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=35)
for bar, val in zip(bars, auc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=8)
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_auc_bar.png"), dpi=150, bbox_inches="tight")
plt.close()

print("[7/7] All plots saved to:", OUTPUT_DIR)
print("\n✅  Done! Files produced:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f"    {f}")

Device : cuda
Model  : ai4bharat/IndicBERTv2-MLM-Sam-TLM

[1/7] Loading data ...
  Classes (13): ['anger', 'boredom', 'empty', 'enthusiasm', 'fun', 'happiness', 'hate', 'love', 'neutral', 'relief', 'sadness', 'surprise', 'worry']
  English rows : 40000
  Assamese rows: 39999
  Train=34000 | Val=6000 | Test(Assamese)=39999


  Class weights: [28.12 17.21  3.72  4.05  1.73  0.59  2.32  0.8   0.36  2.02  0.6   1.41
  0.36]

[2/7] Loading tokenizer ...


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-Sam-TLM
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

  Total params    : 278,441,741
  Trainable params: 278,441,741

[4/7] Training ...
  Epoch 1/5 | Train Loss=2.4464 Acc=0.1995 | Val Loss=2.2540 Acc=0.2850 F1=0.1878
    ✓ Best model saved (Val F1=0.1878)
  Epoch 2/5 | Train Loss=2.2018 Acc=0.2746 | Val Loss=2.2169 Acc=0.2785 F1=0.2071
    ✓ Best model saved (Val F1=0.2071)
  Epoch 3/5 | Train Loss=2.0407 Acc=0.2910 | Val Loss=2.1888 Acc=0.2670 F1=0.2054
  Epoch 4/5 | Train Loss=1.9096 Acc=0.3065 | Val Loss=2.2276 Acc=0.2595 F1=0.2073
    ✓ Best model saved (Val F1=0.2073)
  Epoch 5/5 | Train Loss=1.8067 Acc=0.3178 | Val Loss=2.2567 Acc=0.2603 F1=0.2065

[5/7] Evaluating on Assamese test set ...

  TEST RESULTS ON ASSAMESE DATA
  Accuracy       : 0.2583
  Precision (mac): 0.2273
  Recall (mac)   : 0.2369
  Macro F1       : 0.2053
  ROC-AUC (macro): 0.7422

Per-class Report:

              precision    recall  f1-score   support

       anger       0.00      0.00      0.00       110
     boredom       0.17      0.11      0.13       179
